# Objetivo do trabalho
Uso da VGG16 para detecção de cancer de intestino

### Download do LC25000 através do kaggle

In [ ]:
from google.colab import files
!pip install -q kaggle

In [ ]:
def upload_kaggle_dataset_in_colab():
    files.upload()
    !mkdir -p ~/.kaggle
    !mv kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json

def unzip_kaggle_dataset_in_colab():
    !kaggle datasets download -d andrewmvd/lung-and-colon-cancer-histopathological-images
    !unzip -q lung-and-colon-cancer-histopathological-images.zip

def unzip_support_functions_in_colab():
    !kaggle datasets download -d gabrielcruzvazsantos/colon-detection-support-functions
    !unzip -q colon-detection-support-functions.zip


def get_kaggle_dataset_in_colab():
    upload_kaggle_dataset_in_colab()
    unzip_kaggle_dataset_in_colab()
    unzip_support_functions_in_colab()

In [ ]:
get_kaggle_dataset_in_colab()

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/andrewmvd/lung-and-colon-cancer-histopathological-images
License(s): CC-BY-SA-4.0
 99% 1.75G/1.76G [00:15<00:00, 138MB/s]
100% 1.76G/1.76G [00:15<00:00, 122MB/s]
Dataset URL: https://www.kaggle.com/datasets/gabrielcruzvazsantos/colon-detection-support-functions
License(s): unknown
  0% 0.00/3.86k [00:00<?, ?B/s]
100% 3.86k/3.86k [00:00<00:00, 16.8MB/s]


## Uso de código externo
Esse notebook foi preparado para utilizar código modular python externo que, para seu devido funcionamento, precisa ser feito o upload via

In [ ]:
from get_formatted_datasets import get_formatted_datasets
from get_dataloaders import get_dataloaders
from train_and_test_model import train_model, test_model


In [ ]:
import torch
import torchvision
from abc import ABC, abstractmethod
import matplotlib.pyplot as plt
from tabulate import tabulate
import seaborn as sns


### Datasets e dataLoader

In [ ]:
df_train, df_validation, df_test = get_formatted_datasets()

100%|██████████| 10002/10002 [00:00<00:00, 26797.12it/s]


CSV salvo com sucesso em: nb_lc25000.csv


In [ ]:
dataloader_train, dataloader_validation, dataloader_test = get_dataloaders(df_train, df_validation, df_test)

### Modelo e treinamento
- VGG16
- Canais de entrada - imagem RGB: 3
- Saída - Problema de classificação binária: 2
- Taxa de aprendizado: 1e-2
- Otimizador: Gradiente descendente com Momentaum
- Momentaum: 0.9
- Alpha Schedule: dividir por 10 quando estagnar
- Weight Decay: 5 * 1e-4
- Dropout: 0.5 nas primeiras canadas fc

In [ ]:
in_channels = 3
output_features = 2
learning_rate = 1e-2
optimizer = torch.optim.SGD(torchvision.models.vgg16(num_classes=output_features).parameters(), lr=learning_rate, momentum=0.9, weight_decay=5*1e-4)
alpha_schedule = 0.1


In [ ]:
model = torchvision.models.vgg16(num_classes=output_features)

In [ ]:
model = model.to("cuda")

In [ ]:
print(model.classifier)

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=2, bias=True)
)


In [ ]:
train_losses, val_losses, best_model = train_model(model, dataloader_train, dataloader_validation, optimizer)

Epoch 1: Train Loss = 0.6948 | Val Loss = 0.6927
Epoch 2: Train Loss = 0.6942 | Val Loss = 0.6927
Epoch 3: Train Loss = 0.6923 | Val Loss = 0.6927
Epoch 4: Train Loss = 0.6930 | Val Loss = 0.6927
Epoch 5: Train Loss = 0.6953 | Val Loss = 0.6927
Epoch 6: Train Loss = 0.6948 | Val Loss = 0.6927
Epoch 7: Train Loss = 0.6950 | Val Loss = 0.6927
Epoch 8: Train Loss = 0.6944 | Val Loss = 0.6927
Epoch 9: Train Loss = 0.6944 | Val Loss = 0.6927
Epoch 10: Train Loss = 0.6930 | Val Loss = 0.6927


### Tentativa uso do transferLearning

In [ ]:
model_transfer = torchvision.models.vgg16(pretrained=True)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 74.5MB/s]


In [ ]:
for param in model_transfer.features.parameters():
  param.requires_grad = False

num_ftrs = model_transfer.classifier[6].in_features
model_transfer.classifier[6] = torch.nn.Linear(num_ftrs, output_features)

In [ ]:
model_transfer = model_transfer.to("cuda")
print(model_transfer.classifier)

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=2, bias=True)
)


In [ ]:
train_losses, val_losses, best_model = train_model(model_transfer, dataloader_train, dataloader_validation, optimizer)

Epoch 1: Train Loss = 0.8849 | Val Loss = 0.8375
Epoch 2: Train Loss = 0.8835 | Val Loss = 0.8375
Epoch 3: Train Loss = 0.8880 | Val Loss = 0.8375
Epoch 4: Train Loss = 0.8834 | Val Loss = 0.8375
Epoch 5: Train Loss = 0.8841 | Val Loss = 0.8375
Epoch 6: Train Loss = 0.8808 | Val Loss = 0.8375
Epoch 7: Train Loss = 0.8888 | Val Loss = 0.8375
Epoch 8: Train Loss = 0.8846 | Val Loss = 0.8375
Epoch 9: Train Loss = 0.8883 | Val Loss = 0.8375
Epoch 10: Train Loss = 0.8880 | Val Loss = 0.8375


### Tentativa: aumento do learning rate
De 1e⁻2, para 1e⁻1

In [ ]:
in_channels = 3
output_features = 2
learning_rate = 1e-1
optimizer = torch.optim.SGD(torchvision.models.vgg16(num_classes=output_features).parameters(), lr=learning_rate, momentum=0.9, weight_decay=5*1e-4)
alpha_schedule = 0.1

In [ ]:
model_lr_reduced = torchvision.models.vgg16(num_classes=output_features)
model_lr_reduced = model_lr_reduced.to("cuda")

In [ ]:
train_losses, val_losses, best_model = train_model(model_lr_reduced, dataloader_train, dataloader_validation, optimizer)

Epoch 1: Train Loss = 0.7018 | Val Loss = 0.6957
Epoch 2: Train Loss = 0.7005 | Val Loss = 0.6957
Epoch 3: Train Loss = 0.7018 | Val Loss = 0.6957
Epoch 4: Train Loss = 0.7008 | Val Loss = 0.6957
Epoch 5: Train Loss = 0.7005 | Val Loss = 0.6957
Epoch 6: Train Loss = 0.7008 | Val Loss = 0.6957
Epoch 7: Train Loss = 0.7024 | Val Loss = 0.6957
Epoch 8: Train Loss = 0.6991 | Val Loss = 0.6957
Epoch 9: Train Loss = 0.6996 | Val Loss = 0.6957
Epoch 10: Train Loss = 0.7019 | Val Loss = 0.6957


### Tentativa: diminuir learning rate
De 1e-1 para 1e-4

In [ ]:
in_channels = 3
output_features = 2
learning_rate = 1e-4
optimizer = torch.optim.SGD(torchvision.models.vgg16(num_classes=output_features).parameters(), lr=learning_rate, momentum=0.9, weight_decay=5*1e-4)
alpha_schedule = 0.1

In [ ]:
model_lr_reduced = torchvision.models.vgg16(num_classes=output_features)
model_lr_reduced = model_lr_reduced.to("cuda")

In [ ]:
train_losses, val_losses, best_model = train_model(model_lr_reduced, dataloader_train, dataloader_validation, optimizer)

Epoch 1: Train Loss = 0.6942 | Val Loss = 0.6933
Epoch 2: Train Loss = 0.6952 | Val Loss = 0.6933
Epoch 3: Train Loss = 0.6956 | Val Loss = 0.6933
Epoch 4: Train Loss = 0.6951 | Val Loss = 0.6933
Epoch 5: Train Loss = 0.6960 | Val Loss = 0.6933
Epoch 6: Train Loss = 0.6950 | Val Loss = 0.6933
Epoch 7: Train Loss = 0.6952 | Val Loss = 0.6933
Epoch 8: Train Loss = 0.6949 | Val Loss = 0.6933
Epoch 9: Train Loss = 0.6961 | Val Loss = 0.6933
Epoch 10: Train Loss = 0.6952 | Val Loss = 0.6933


### Tentativa: Uso de batchs e early stopping
Uso ideal de batchs para vgg16: 256 ou 128

In [ ]:
dataloader_train, dataloader_validation, dataloader_test = get_dataloaders(df_train, df_validation, df_test, 128)

In [ ]:
in_channels = 3
output_features = 2
learning_rate = 1e-2
optimizer = torch.optim.SGD(torchvision.models.vgg16(num_classes=output_features).parameters(), lr=learning_rate, momentum=0.9, weight_decay=5*1e-4)
alpha_schedule = 0.1
batch_size = 128
patience = 5

In [ ]:
model_batch = torchvision.models.vgg16(num_classes=output_features)
model_batch = model_batch.to("cuda")

In [ ]:
train_losses, val_losses, best_model = train_model(model_batch, dataloader_train, dataloader_validation, optimizer)

Epoch 1/10 | Train Loss: 0.6953 | Val Loss: 0.6933 | Sem melhora: 0/5
Epoch 2/10 | Train Loss: 0.6952 | Val Loss: 0.6933 | Sem melhora: 1/5
Epoch 3/10 | Train Loss: 0.6952 | Val Loss: 0.6933 | Sem melhora: 2/5
Epoch 4/10 | Train Loss: 0.6963 | Val Loss: 0.6933 | Sem melhora: 3/5
Epoch 5/10 | Train Loss: 0.6963 | Val Loss: 0.6933 | Sem melhora: 4/5
Epoch 6/10 | Train Loss: 0.6955 | Val Loss: 0.6933 | Sem melhora: 5/5

Early stopping ativado! Melhor Val Loss: 0.6933


In [ ]:
train_losses, val_losses, best_model = train_model(model_batch, dataloader_train, dataloader_validation, optimizer, 30, 10)

Epoch 1/30 | Train Loss: 0.6959 | Val Loss: 0.6933 | Sem melhora: 0/10
Epoch 2/30 | Train Loss: 0.6952 | Val Loss: 0.6933 | Sem melhora: 1/10
Epoch 3/30 | Train Loss: 0.6961 | Val Loss: 0.6933 | Sem melhora: 2/10
Epoch 4/30 | Train Loss: 0.6954 | Val Loss: 0.6933 | Sem melhora: 3/10
Epoch 5/30 | Train Loss: 0.6959 | Val Loss: 0.6933 | Sem melhora: 4/10
Epoch 6/30 | Train Loss: 0.6960 | Val Loss: 0.6933 | Sem melhora: 5/10
Epoch 7/30 | Train Loss: 0.6952 | Val Loss: 0.6933 | Sem melhora: 6/10
Epoch 8/30 | Train Loss: 0.6959 | Val Loss: 0.6933 | Sem melhora: 7/10
Epoch 9/30 | Train Loss: 0.6958 | Val Loss: 0.6933 | Sem melhora: 8/10
Epoch 10/30 | Train Loss: 0.6957 | Val Loss: 0.6933 | Sem melhora: 9/10
Epoch 11/30 | Train Loss: 0.6950 | Val Loss: 0.6933 | Sem melhora: 10/10

Early stopping ativado! Melhor Val Loss: 0.6933


### Tentativa: Batch + transfer Learning + Early stopping

In [ ]:
dataloader_train, dataloader_validation, dataloader_test = get_dataloaders(df_train, df_validation, df_test, 128)

In [ ]:
in_channels = 3
output_features = 2
learning_rate = 1e-2
optimizer = torch.optim.SGD(torchvision.models.vgg16(num_classes=output_features).parameters(), lr=learning_rate, momentum=0.9, weight_decay=5*1e-4)
alpha_schedule = 0.1
batch_size = 128
patience = 5

In [ ]:
model_transfer = torchvision.models.vgg16(pretrained=True)

In [ ]:
for param in model_transfer.features.parameters():
  param.requires_grad = False

num_ftrs = model_transfer.classifier[6].in_features
model_transfer.classifier[6] = torch.nn.Linear(num_ftrs, output_features)

In [ ]:
model_transfer = model_transfer.to("cuda")


In [ ]:
train_losses, val_losses, best_model = train_model(model_transfer, dataloader_train, dataloader_validation, optimizer, 30, 10)

Epoch 1/30 | Train Loss: 0.7552 | Val Loss: 0.7150 | Sem melhora: 0/10
Epoch 2/30 | Train Loss: 0.7512 | Val Loss: 0.7150 | Sem melhora: 1/10
Epoch 3/30 | Train Loss: 0.7594 | Val Loss: 0.7150 | Sem melhora: 2/10
Epoch 4/30 | Train Loss: 0.7536 | Val Loss: 0.7150 | Sem melhora: 3/10
Epoch 5/30 | Train Loss: 0.7497 | Val Loss: 0.7150 | Sem melhora: 4/10
Epoch 6/30 | Train Loss: 0.7554 | Val Loss: 0.7150 | Sem melhora: 5/10
Epoch 7/30 | Train Loss: 0.7527 | Val Loss: 0.7150 | Sem melhora: 6/10
Epoch 8/30 | Train Loss: 0.7512 | Val Loss: 0.7150 | Sem melhora: 7/10
Epoch 9/30 | Train Loss: 0.7537 | Val Loss: 0.7150 | Sem melhora: 8/10
Epoch 10/30 | Train Loss: 0.7502 | Val Loss: 0.7150 | Sem melhora: 9/10


#### imprimindo resultados do treinamento e validação

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="Treinamento")
plt.plot(val_losses, label="Validação")
plt.title("Loss por Época")
plt.xlabel("Época")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

### Testando modelo

In [ ]:
acc, precision, recall, f1, cm = test_model(best_model, "cuda", dataloader_test)

In [ ]:
headers = ["Métrica", "Valor"]
table = [
["Acurácia", f"{acc:.4f}"],
["Precisão (weighted)", f"{precision:.4f}"],
["Recall (weighted)", f"{recall:.4f}"],
["F1-score (weighted)", f"{f1:.4f}"],
]
print(tabulate(table, headers=headers, tablefmt="fancy_grid"))

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title("Matriz de Confusão")
plt.xlabel("Predito")
plt.ylabel("Real")
plt.show()